## Adaptive RAG

In [1]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb as apply_rotary_pos_emb
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
import json
import random
import os
import glob
import faiss
from sentence_transformers import SentenceTransformer
print('Done.')

Done.


In [2]:
from utils import get_hook_q, get_hook_k, get_res_hook
from utils import get_p_tot_log, get_head_magnitudes, get_logit_feats, extract_features
print('Done.')

Done.


In [3]:
from model import HallucinationDetector
from model import stream_openai_lambada, generate_hallucination_dataset
from model import train_hallucination_detector
print('Done.')

Done.


In [4]:
from evals import evaluate_detector, run_kfold_evaluation
from evals import plot_trajectory_comparison, plot_trajectory_grid_scaled, plot_evidence_distribution
print('Done.')

Done.


In [5]:
# from utils import get_search_results, build_rag_prompt, run_zero_shot_baseline
# from utils import score_response, evaluate_zero_shot_full_text
# print('Done.')

### Build RAG database

In [6]:
current_dir = os.getcwd()

In [7]:
# Load the model locally
local_model_path = os.path.join(current_dir, "all-MiniLM-L6-v2")
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embed_model = SentenceTransformer(local_model_path)

sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embed_model.encode(sentences)

print(f"Embeddings Shape: {embeddings.shape}")
# print(embeddings)

Embeddings Shape: (2, 384)


In [8]:
# Extract context from TriviaQA JSON
def load_trivia_snippets(data_folder, limit=500):
    snippets = []
    # Path to the JSON metadata
    json_path = os.path.join(data_folder, 'qa', 'wikipedia-dev.json')
    # Path to the folder containing the raw text files
    evidence_folder = os.path.join(data_folder, 'evidence', 'wikipedia')

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    print(f"Processing {limit} questions to find text files...")
    
    for item in data.get('Data', [])[:limit]:
        if 'EntityPages' in item:
            for page in item['EntityPages']:
                filename = page.get('Filename')
                if filename:
                    # Construct the path to the actual .txt file
                    txt_path = os.path.join(evidence_folder, filename)
                    
                    if os.path.exists(txt_path):
                        with open(txt_path, 'r', encoding='utf-8', errors='ignore') as tf:
                            content = tf.read()
                            # Optional: Take only the first 2000 characters to keep vectors clean
                            snippets.append(content[:2000])
                    else:
                        # Debugging: Print if a file is missing
                        pass 

    return snippets

# Run the loader
trivia_data_path = os.path.join(current_dir, "data\\triviaqa-rc")
all_snippets = load_trivia_snippets(trivia_data_path)
print(f"Total snippets successfully read from disk: {len(all_snippets)}")

Processing 500 questions to find text files...
Total snippets successfully read from disk: 846


In [9]:
# Convert text to vectors
print(f"Encoding {len(all_snippets)} snippets... this may take a minute.")
embeddings = embed_model.encode(all_snippets, show_progress_bar=True)
embeddings_np = np.array(embeddings).astype('float32')

# Initialize the index
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add the vectors to the index
index.add(embeddings_np)

print(f"FAISS Index ready with {index.ntotal} vectors.")

# Save the index and the snippets list locally
faiss.write_index(index, "trivia_kb.index")
with open("snippets_map.json", "w", encoding='utf-8') as f:
    json.dump(all_snippets, f)

Encoding 846 snippets... this may take a minute.


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

FAISS Index ready with 846 vectors.


### RAG

In [10]:
def get_search_results(embed_model, snippets, query, index, k=3):
    # Vectorize the new query
    query_vector = embed_model.encode([query]).astype('float32')
    
    # Search the index
    distances, indices = index.search(query_vector, k)
    
    # Pull the actual text strings
    retrieved_context = [all_snippets[i] for i in indices[0]]
    
    return retrieved_context

In [11]:
def build_rag_prompt(question, context_list):
    # Combine the snippets into one string
    context_str = "\n\n".join([f"Source {i+1}: {text}" for i, text in enumerate(context_list)])
    
    # Create the template
    prompt = f"""Use the following pieces of context to answer the question at the end. 
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    CONTEXT:
    {context_str}

    QUESTION: 
    {question}

    ANSWER:"""
    
    return prompt

In [12]:
# Load tokenizer and model

model_path = os.path.join(current_dir, "TinyLlama-1.1B-Chat-v1.0")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    local_files_only=True
)

print(f"Model loaded onto: {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded onto: cuda:0


In [13]:
# The User Question
user_query = "What is the capital of China?"
# user_query = " The capital of China is "

inputs_no_rag = tokenizer(user_query, return_tensors="pt").to(model.device)
outputs_no_rag = model.generate(**inputs_no_rag, max_new_tokens=50)
answer_no_rag = tokenizer.decode(outputs_no_rag[0], skip_special_tokens=True)
answer_no_rag

'What is the capital of China?'

In [14]:
# Get the "Open Book" info
context_snippets = get_search_results(embed_model, all_snippets, user_query, index, k=2)

# Create the final prompt
final_prompt = build_rag_prompt(user_query, context_snippets)

# Feed to your TinyLlama
inputs = tokenizer(final_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

prompt_length = inputs.input_ids.shape[-1] # Get length of input prompt
new_tokens = outputs[0][prompt_length:] # Slice output tensor to take everything AFTER that length
answer_only = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(user_query)
print(answer_only)

What is the capital of China?
Beijing


### Compare performance with/without RAG

In [15]:
def run_zero_shot_baseline(data_folder, num_questions=5):
    # Load the metadata to get the questions
    json_path = os.path.join(data_folder, 'qa', 'wikipedia-dev.json')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    questions_data = data.get('Data', [])[:num_questions]
    
    print(f"--- Running {num_questions} Zero-Shot Tests ---")
    
    for i, item in enumerate(questions_data):
        question = item['Question']
        
        # We use a Q&A format to help the base model focus
        prompt = f"Question: {question}\nAnswer:"
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Generate
        outputs = model.generate(
            **inputs, 
            max_new_tokens=30, 
            do_sample=False, # Keep it deterministic
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Slice to get only the new tokens
        new_tokens = outputs[0][inputs.input_ids.shape[-1]:]
        raw_answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        # Clean up: stop at the first newline or period
        clean_answer = raw_answer.split('\n')[0].split('.')[0]
        
        print(f"\n[{i+1}] Q: {question}")
        print(f"    A: {clean_answer}")

run_zero_shot_baseline(trivia_data_path, num_questions=10)

--- Running 10 Zero-Shot Tests ---

[1] Q: Which Lloyd Webber musical premiered in the US on 10th December 1993?
    A: The Phantom of the Opera

[2] Q: Who was the next British Prime Minister after Arthur Balfour?
    A: The next British Prime Minister after Arthur Balfour was Winston Churchill

[3] Q: Who had a 70s No 1 hit with Kiss You All Over?
    A: The Kinks

[4] Q: What claimed the life of singer Kathleen Ferrier?
    A: Kathleen Ferrier was killed in a car accident in 1951

[5] Q: Which actress was voted Miss Greenwich Village in 1942?
    A: Marjorie Main

[6] Q: What was the name of Michael Jackson's autobiography written in 1988?
    A: "Michael Jackson: The Life and Times of a King"

[7] Q: Which volcano in Tanzania is the highest mountain in Africa?
    A: Mount Kilimanjaro, located in Tanzania, is the highest mountain in Africa

[8] Q: The flag of Libya is a plain rectangle of which color?
    A: The flag of Libya is a plain rectangle of which color is red

[9] Q: Of wh

In [16]:
def score_response(llm_output, gold_aliases):
    """
    Checks if any of the correct aliases appear in the LLM's generated text.
    """
    # Standardize to lowercase for fairness
    prediction = llm_output.lower()
    
    # Check each alias
    for alias in gold_aliases:
        if alias.lower() in prediction:
            return 1.0  # Perfect match found in the sentence
            
    return 0.0  # No correct alias found

In [17]:
def evaluate_zero_shot_full_text(model, embed_model, tokenizer, data_folder, num_questions=10, use_rag=False, k=3, show_text=True):
    json_path = os.path.join(data_folder, 'qa', 'wikipedia-dev.json')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    questions_data = data.get('Data', [])[:num_questions]
    scores = []
    
    if show_text:
        print(f"--- Full Text Evaluation ({num_questions} Questions) ---\n")

    for i, item in enumerate(tqdm(questions_data)):
        question = item['Question']
        gold_aliases = item['Answer']['Aliases']
        
        # Generation
        prompt = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Determine the Prompt String
        if use_rag:
            current_k = k
            while current_k > 0:
                # Retrieve current_k snippets
                context_snippets = get_search_results(embed_model, question, k=current_k)
                prompt_text = build_rag_prompt(question, context_snippets)
                
                # Check token length
                temp_ids = tokenizer.encode(prompt_text)
                if len(temp_ids) <= 1800:
                    break # It fits!
                
                # If too long, reduce k and try again
                current_k -= 1
                print(f"TRIMMING: Question {i+1} too long, reducing k to {current_k}")
            
            # Final fallback: if even k=1 is too long, truncate the text of the first snippet
            if current_k == 0:
                context_snippets = [get_search_results(embed_model, question, k=1)[0][:500]]
                prompt_text = build_rag_prompt(question, context_snippets)
        else:
            prompt_text = f"Question: {question}\nAnswer:"
        
        # Tokenize the chosen prompt
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens=50, # Increased slightly to capture full sentences
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
        
        prompt_length = inputs.input_ids.shape[-1]
        answer_raw = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()
        
        # Clean to the first actual thought (stopping at double newlines)
        answer_clean = answer_raw.split('\n\n')[0].strip()
        
        # Scoring
        is_correct = any(alias.lower() in answer_clean.lower() for alias in gold_aliases)
        score = 1.0 if is_correct else 0.0
        scores.append(score)
        
        if show_text:
            # Display Results
            print(f"TEST #{i+1}")
            print(f"QUESTION: {question}")
            print(f"MODEL OUTPUT: {answer_clean}")
            print(f"GOLD ALIASES: {', '.join(gold_aliases)}")
            print(f"MATCH: {'✅ YES' if is_correct else '❌ NO'}")
            print("-" * 50)

    accuracy = (sum(scores) / len(scores)) * 100
    print(f"FINAL ACCURACY: {accuracy:.2f}%")

In [18]:
# Run the full-text loop
evaluate_zero_shot_full_text(model, embed_model, tokenizer, trivia_data_path, num_questions=1, use_rag=False, show_text=True)

--- Full Text Evaluation (1 Questions) ---



100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.24s/it]

TEST #1
QUESTION: Which Lloyd Webber musical premiered in the US on 10th December 1993?
MODEL OUTPUT: The Phantom of the Opera.
Question: Which Lloyd Webber musical premiered in the US on 10th December 1993?
Question: Which Lloyd Webber musical premiered in the US on 10th
GOLD ALIASES: Sunset Blvd, West Sunset Boulevard, Sunset Boulevard, Sunset Bulevard, Sunset Blvd.
MATCH: ❌ NO
--------------------------------------------------
FINAL ACCURACY: 0.00%


In [19]:
# Run the full-text loop
evaluate_zero_shot_full_text(model, embed_model, tokenizer, trivia_data_path, num_questions=100, use_rag=False, show_text=False)

  5%|████                                                                              | 5/100 [00:09<03:04,  1.95s/it]


KeyboardInterrupt: 

In [ ]:
# Run the full-text loop
evaluate_zero_shot_full_text(model, embed_model, tokenizer, trivia_data_path, num_questions=100, use_rag=True, k=3, show_text=False)

### Train Hallucination Detector

In [ ]:
def get_random_trivia_entries(data_folder, num_samples=5000, seed=2026):
    """
    Modular function to grab N random samples from the TriviaQA training set.
    """
    json_path = os.path.join(data_folder, 'qa', 'wikipedia-train.json')
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_entries = data.get('Data', [])
    
    # Ensure we don't try to sample more than exists
    num_samples = min(num_samples, len(all_entries))
    
    random.seed(seed)
    sampled_entries = random.sample(all_entries, num_samples)
    
    print(f"Successfully sampled {len(sampled_entries)} entries from a total of {len(all_entries)}.")
    
    return sampled_entries

In [ ]:
def generate_trivia_features(model, embed_model, tokenizer, sampled_entries, milestones=[11, 21], use_rag=False, limit=None):
    """
    Processes a pre-sampled list of entries up to a specified limit.
    """
    X, y = [], []
    saved = {}
    
    # Use the limit if provided, otherwise process the whole list
    entries_to_process = sampled_entries[:limit] if limit is not None else sampled_entries

    print(f"Generating {'RAG' if use_rag else 'Zero-Shot'} features for {len(entries_to_process)} samples...")

    for i, item in enumerate(tqdm(entries_to_process)):
        question = item['Question']
        gold_aliases = item['Answer']['Aliases']
        
        saved.clear()
        handles = []

        # Register hooks
        for idx, layer in enumerate(model.model.layers):
            handles.append(layer.register_forward_hook(get_res_hook(idx, saved)))
            if idx in milestones:
                handles.append(layer.self_attn.q_proj.register_forward_hook(get_hook_q(f"lyr_{idx}", saved)))
                handles.append(layer.self_attn.k_proj.register_forward_hook(get_hook_k(f"lyr_{idx}", saved)))

        # Build Prompt
        if use_rag:
            context = get_search_results(embed_model, question, k=2) 
            prompt = build_rag_prompt(question, context)
        else:
            prompt = f"Question: {question}\nAnswer:"
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Forward Pass & Generation
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            gen_outputs = model.generate(
                **inputs, 
                max_new_tokens=30, 
                do_sample=False, 
                repetition_penalty=1.2,
                pad_token_id=tokenizer.eos_token_id
            )

        # Feature Extraction
        feat_vec = extract_features(model, outputs, saved, milestones)
        
        # Append Mode Flag (Feature #239)
        mode_flag = torch.tensor([1.0 if use_rag else 0.0]).to(feat_vec.device)
        feat_vec = torch.cat([feat_vec, mode_flag])

        # Labeling
        prompt_len = inputs.input_ids.shape[-1]
        decoded = tokenizer.decode(gen_outputs[0][prompt_len:], skip_special_tokens=True).strip()
        answer_clean = decoded.split('\n')[0].strip()
        
        is_correct = any(alias.lower() in answer_clean.lower() for alias in gold_aliases)
        X.append(feat_vec.cpu())
        y.append(1.0 if is_correct else 0.0)

        # Cleanup hooks
        for h in handles:
            h.remove()

    # Convert to Tensors and Shuffle
    X_tensor = torch.stack(X).float()
    y_tensor = torch.tensor(y).float()
    
    indices = torch.randperm(len(X_tensor))
    return X_tensor[indices], y_tensor[indices]

In [ ]:
def generate_sequential_training_data(model, embed_model, tokenizer, sampled_entries, save_path, start_idx=0, num_to_process=100):
    """
    Processes a specific slice of the sampled_entries list.
    """
    # Slice the entries for this specific run
    end_idx = start_idx + num_to_process
    current_chunk = sampled_entries[start_idx:end_idx]
    
    if not current_chunk:
        print("No more entries to process in this range!")
        return None, None

    print(f"--- Processing Slice: {start_idx} to {end_idx} ---")

    # Run Zero-Shot on this chunk
    X_zs, y_zs = generate_trivia_features(
        model, embed_model, tokenizer, 
        current_chunk, use_rag=False
    )
    
    # Run RAG on the SAME chunk
    X_rag, y_rag = generate_trivia_features(
        model, embed_model, tokenizer, 
        current_chunk, use_rag=True
    )
    
    # Combine
    X_chunk = torch.cat([X_zs, X_rag], dim=0)
    y_chunk = torch.cat([y_zs, y_rag], dim=0)
    
    # Save this specific chunk (numbered for easy tracking)
    chunk_filename = f"{save_path}_chunk_{start_idx}_{end_idx}.pt"
    torch.save({'X': X_chunk, 'y': y_chunk}, chunk_filename)
    
    print(f"Chunk saved to {chunk_filename}")
    return X_chunk, y_chunk

In [ ]:
save_path = "data\\trivia_hallucination_combined.pt"

# Get subset of full dataset
# sampled_entries = get_random_trivia_entries(trivia_data_path, num_samples=5000)

# Get training data
X1, y1 = generate_sequential_training_data(model, embed_model, tokenizer, sampled_entries, save_path, start_idx=4500, num_to_process=500)

In [27]:
def merge_trivia_chunks(data_dir="data", save_path="data/trivia_hallucination_final.pt"):
    # Look for files ending in _chunk_START_END
    search_pattern = os.path.join(data_dir, "*_chunk_*_*")
    chunk_files = sorted(glob.glob(search_pattern))
    
    if not chunk_files:
        print(f"No chunks found in {data_dir} matching the pattern.")
        return None, None
    
    all_X, all_y = [], []
    print(f"Found {len(chunk_files)} chunks. Merging...")

    # Load each chunk
    for f in chunk_files:
        try:
            data = torch.load(f)
            # Ensure we are grabbing the right keys
            all_X.append(data['X'])
            all_y.append(data['y'])
            print(f"Successfully loaded {os.path.basename(f)}: {data['X'].shape[0]} samples")
        except Exception as e:
            print(f"Error loading {f}: {e}")

    # Concatenate all tensors
    X_final = torch.cat(all_X, dim=0)
    y_final = torch.cat(all_y, dim=0)

    # Final Save
    torch.save({
        'X': X_final,
        'y': y_final,
        'metadata': {
            'total_samples': len(X_final),
            'feature_dim': X_final.shape[1],
            'chunk_count': len(chunk_files)
        }
    }, save_path)

    print(f"\n--- Merge Complete ---")
    print(f"Total Samples: {X_final.shape[0]}")
    print(f"Features per Sample: {X_final.shape[1]}")
    print(f"Final file saved at: {save_path}")
    
    return X_final, y_final

# Run it
X_master, y_master = merge_trivia_chunks()

Found 1 chunks. Merging...
Successfully loaded trivia_hallucination_combined.pt_chunk_4500_5000.pt: 1000 samples

--- Merge Complete ---
Total Samples: 1000
Features per Sample: 239
Final file saved at: data/trivia_hallucination_final.pt


C:\Users\Pracioppo\AppData\Local\Temp\ipykernel_26092\1836782065.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f)


### Train and evaluate detector performance

In [24]:
# Load data from file
checkpoint = torch.load("data/trivia_hallucination_final.pt")
X_tot, y_tot = checkpoint['X'], checkpoint['y']

# Prune Features (Removing 182:236)
# i.e. we keep only the top 10 logits
X_tot = torch.cat([X_tot[:, :182], X_tot[:, 237:]], dim=1)

input_dim = X_tot.shape[1]
print(input_dim)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

184


C:\Users\Pracioppo\AppData\Local\Temp\ipykernel_26092\2426925181.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("data/trivia_hallucination_final

In [ ]:
results_trivia = run_kfold_evaluation(
    X_raw=X_tot,       # Your shuffled 10k tensor
    y_raw=y_tot,       # Your shuffled labels
    model=HallucinationDetector,
    k_folds=5, 
    threshold=0.5,
    input_dim=input_dim,        # Updated for the RAG flag
    hidden_dim=64,        # Slightly wider for 10k samples
    epochs=100,           # 100 is usually enough for 10k samples
    batch_size=64,        # Larger batch for stability
    device=device,
    pos_weight=2.0,       # Adjusted for TriviaQA balance
)

In [ ]:
results_trivia = run_kfold_evaluation(
    X_raw=X_tot,       # Your shuffled 10k tensor
    y_raw=y_tot,       # Your shuffled labels
    model=HallucinationDetector,
    k_folds=5, 
    threshold=0.8,
    input_dim=input_dim,        # Updated for the RAG flag
    hidden_dim=64,        # Slightly wider for 10k samples
    epochs=100,           # 100 is usually enough for 10k samples
    batch_size=64,        # Larger batch for stability
    device=device,
    pos_weight=2.0,       # Adjusted for TriviaQA balance
)

In [ ]:
def run_kfold_evaluation_multisplit(X_raw, y_raw, k_folds=5, input_dim=239, hidden_dim=64, batch_size=32, epochs=100, pos_weight=2.0):
    indices = torch.randperm(len(X_raw))
    fold_size = len(X_raw) // k_folds
    fold_results = []  # Changed to a list of dicts for easier iteration

    for fold in range(k_folds):
        print(f"\n--- FOLD {fold + 1}/{k_folds} ---")
        
        test_idx = indices[fold * fold_size : (fold + 1) * fold_size]
        train_idx = torch.cat((indices[:fold * fold_size], indices[(fold + 1) * fold_size:]))
        
        X_train_raw, y_train = X_raw[train_idx], y_raw[train_idx].to(device)
        X_test_raw, y_test = X_raw[test_idx], y_raw[test_idx].to(device)

        X_mean, X_std = X_train_raw.mean(dim=0), X_train_raw.std(dim=0) + 1e-6
        X_train = ((X_train_raw - X_mean) / X_std).to(device)
        X_test = ((X_test_raw - X_mean) / X_std).to(device)

        detector = HallucinationDetector(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
        train_hallucination_detector(detector, X_train, y_train, epochs=epochs, batch_size=batch_size, pos_weight=pos_weight)
        
        detector.eval()
        with torch.no_grad():
            # Get masks for the slices
            is_rag_test = X_test_raw[:, -1] == 1.0
            is_zs_test = X_test_raw[:, -1] == 0.0

            def get_f1(X, y):
                if len(y) == 0: return 0.0
                logits = detector(X)
                preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
                return f1_score(y.cpu().numpy(), preds, pos_label=0)

            # Store results for this fold
            fold_results.append({
                'combined_f1': get_f1(X_test, y_test),
                'zero_shot_f1': get_f1(X_test[is_zs_test], y_test[is_zs_test]),
                'rag_f1': get_f1(X_test[is_rag_test], y_test[is_rag_test])
            })
            
            print(f"ZS F1: {fold_results[-1]['zero_shot_f1']:.4f} | RAG F1: {fold_results[-1]['rag_f1']:.4f}")

    return fold_results

In [ ]:
# 1. Define hyperparameters
# We use 239 because the last column of X_master is the 0/1 RAG flag
input_dim = X_tot.shape[1] 
hidden_dim = 64
k_folds = 5
epochs = 100

# 2. Execute the Multi-Split Evaluation
# This will train on the combined data but report ZS, RAG, and Combined scores per fold
cv_results = run_kfold_evaluation_multisplit(
    X_raw=X_tot, 
    y_raw=y_tot, 
    k_folds=k_folds, 
    input_dim=input_dim, 
    hidden_dim=hidden_dim
)

# 3. (Optional) Quick check of the final averages across folds
avg_zs_f1 = np.mean([f['zero_shot_f1'] for f in cv_results])
avg_rag_f1 = np.mean([f['rag_f1'] for f in cv_results])

print(f"\nFINAL COMPARISON:")
print(f"Average Zero-Shot Hallucination F1: {avg_zs_f1:.4f}")
print(f"Average RAG-Only Hallucination F1: {avg_rag_f1:.4f}")

### Reflective RAG (RRAG)

In [28]:
# Shuffle the entire 10k set to ensure an even distribution
indices = torch.randperm(len(X_tot))
X_shuffled = X_tot[indices]
y_shuffled = y_tot[indices]

# Define the Split (9k Train / 1k Test)
train_size = 9000
X_train_raw = X_shuffled[:train_size]
y_train = y_shuffled[:train_size].to(device)

X_test_raw = X_shuffled[train_size:]
y_test = y_shuffled[train_size:].to(device)

# Calculate Normalization Stats ON TRAIN ONLY
# We use these same stats to scale the Test set to prevent leakage
X_mean = X_train_raw.mean(dim=0)
X_std = X_train_raw.std(dim=0) + 1e-6

# Scale both sets
X_train = ((X_train_raw - X_mean) / X_std).to(device)
X_test = ((X_test_raw - X_mean) / X_std).to(device)

# Final Verification
print(f"--- Data Split Complete ---")
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print(f"Hallucination Rate (Test): {(y_test == 0).sum().item() / len(y_test):.2%}")

# Verification: The first 500 of the test set are ZS, the last 500 are RAG
mid_test = 500
print(f"Verified ZS Accuracy (Test):  {y_test[:mid_test].mean():.2%}")
print(f"Verified RAG Accuracy (Test): {y_test[mid_test:].mean():.2%}")

--- Data Split Complete ---
Train Shape: torch.Size([1000, 184]) | Test Shape: torch.Size([0, 184])


ZeroDivisionError: division by zero

In [29]:
# # Define the Test Set as the last 1000 rows
# test_size = 1000
# X_test_raw = X_master[-test_size:]
# y_test = y_master[-test_size:].to(device)

# flags_zs = X_test_raw[:500, -1]
# flags_rag = X_test_raw[500:, -1]

# print(f"ZS section flag mean: {flags_zs.mean().item()}") # Target: 0.0
# print(f"RAG section flag mean: {flags_rag.mean().item()}") # Target: 1.0

# # Define the Training Set as everything else
# X_train_raw = X_master[:-test_size]
# y_train = y_master[:-test_size].to(device)

# # Shuffle ONLY the training data
# train_indices = torch.randperm(len(X_train_raw))
# X_train_raw = X_train_raw[train_indices]
# y_train = y_train[train_indices]

# # Calculate Normalization Stats on Train Only
# X_mean = X_train_raw.mean(dim=0)
# X_std = X_train_raw.std(dim=0) + 1e-6

# # Scale both sets
# X_train = ((X_train_raw - X_mean) / X_std).to(device)
# X_test = ((X_test_raw - X_mean) / X_std).to(device)

# print(f"--- Chunk-Stacked Split Complete ---")
# print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

# # Verification: The first 500 of the test set are ZS, the last 500 are RAG
# mid_test = 500
# print(f"Verified ZS Accuracy (Test):  {y_test[:mid_test].mean():.2%}")
# print(f"Verified RAG Accuracy (Test): {y_test[mid_test:].mean():.2%}")

ZS section flag mean: 0.0
RAG section flag mean: 1.0
--- Chunk-Stacked Split Complete ---
Train Shape: torch.Size([0, 239]) | Test Shape: torch.Size([1000, 239])
Verified ZS Accuracy (Test):  37.80%
Verified RAG Accuracy (Test): 35.40%


C:\Users\Pracioppo\AppData\Local\Temp\ipykernel_26092\3962694926.py:23: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\ReduceOps.cpp:1823.)
  X_std = X_train_raw.std(dim=0) + 1e-6


In [ ]:
input_dim = X_train.shape[1] 
detector = HallucinationDetector(input_dim=input_dim, hidden_dim=64).to(device)

# Automatic pos_weight calculation 
# (Ratio of Hallucinations to Correct answers)
num_hallucinations = (y_train == 0).sum().item()
num_correct = (y_train == 1).sum().item()
# weight = num_hallucinations / max(num_correct, 1)
weight = 2.0

print(f"Training with pos_weight: {weight:.2f}")

losses = train_hallucination_detector(
    detector, 
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=32, 
    lr=5e-4, 
    pos_weight=weight
)

In [ ]:
def evaluate_detector(detector, X_test, y_test, X_test_raw, threshold=0.7):
    """
    Evaluates the trained detector on the held-out test set.
    Separates results by Zero-Shot and RAG modes to check for 'Apathy'.
    """
    detector.eval()
    results = {}
    
    with torch.no_grad():
        # 1. Forward pass
        logits = detector(X_test)
        probs = torch.sigmoid(logits)
        
        # 2. Thresholding
        preds = (probs >= threshold).float().cpu().numpy()
        y_true = y_test.cpu().numpy()
        
        # 3. Global Metrics
        results['global_f1_hallucination'] = f1_score(y_true, preds, pos_label=0)
        results['global_accuracy'] = (preds == y_true).mean()

        # 4. Slice by Mode (Feature #183 is the RAG flag in the pruned 184-set)
        # Using X_test_raw because it's not standardized (clean 0.0 or 1.0)
        is_rag = (X_test_raw[:, -1] == 1.0)
        is_zs = (X_test_raw[:, -1] == 0.0)

        results['zs_f1_hallucination'] = f1_score(y_true[is_zs], preds[is_zs], pos_label=0)
        results['rag_f1_hallucination'] = f1_score(y_true[is_rag], preds[is_rag], pos_label=0)
        
        # 5. Print the Report
        print(f"\n" + "="*30)
        print(f" DETECTOR EVALUATION (Threshold: {threshold})")
        print(f"="*30)
        print(classification_report(y_true, preds, target_names=['Hallucination (0)', 'Correct (1)']))
        
        print(f"--- Mode Breakdown ---")
        print(f"Zero-Shot F1: {results['zs_f1_hallucination']:.4f}")
        print(f"RAG-Only F1:  {results['rag_f1_hallucination']:.4f}")
        print(f"F1 Delta:     {results['rag_f1_hallucination'] - results['zs_f1_hallucination']:.4f}")
        
        # 6. Confusion Matrix for the "Safety" check
        cm = confusion_matrix(y_true, preds)
        print(f"\n--- Confusion Matrix ---")
        print(f"Caught Hallucinations (TN): {cm[0,0]}")
        print(f"Missed Hallucinations (FN): {cm[0,1]}")
        print(f"False Alarms on Truth (FP): {cm[1,0]}")
        print(f"Verified Truths       (TP): {cm[1,1]}")

    return results

# Run the eval
test_metrics = evaluate_detector(detector, X_test, y_test, X_test_raw, threshold=0.5)

In [ ]:
# def reflective_rag_generate(query, X_mean, X_std, k=3, max_iters=3, threshold=0.8):
#     best_answer = None
#     max_conf = -1.0
    
#     for i in range(max_iters + 1):  # +1 to include Zero-Shot as the first attempt
#         # 1. Get Context (None for iter 0, then increments of k)
#         context = None if i == 0 else retrieve_docs(query, limit=k, offset=(i-1)*k)
        
#         # 2. Generate and Extract Features (Pruned to 184)
#         answer, raw_features = tinyllama_generate_with_features(query, context)
        
#         # 3. Standardize using the FINAL training stats
#         norm_features = (raw_features - X_mean) / X_std
        
#         # 4. Predict Confidence
#         detector.eval()
#         with torch.no_grad():
#             logit = detector(norm_features.to(device))
#             conf = torch.sigmoid(logit).item()
        
#         print(f"Iter {i} | Conf: {conf:.4f} | Answer: {answer[:50]}...")

#         # 5. The "Reflective" Decision
#         if conf >= threshold:
#             return answer, conf, "Confident"
            
#         if conf > max_conf:
#             max_conf = conf
#             best_answer = answer
            
#     # If we never hit the threshold
#     return best_answer, max_conf, "Max_Iters_Reached"

In [ ]:
def simulate_system_efficiency(X_test_raw, detector, X_mean, X_std, threshold=0.8):
    # Costs (Arbitrary units)
    COST_ZS = 1
    COST_RAG = 5
    
    total_cost_adaptive = 0
    total_samples = len(X_test_raw)
    rag_triggers = 0
    
    detector.eval()
    with torch.no_grad():
        # Prepare Zero-Shot samples from your test set
        is_zs = (X_test_raw[:, -1] == 0.0)
        X_zs_raw = X_test_raw[is_zs]
        
        # Normalize
        X_zs_scaled = (X_zs_raw - X_mean) / X_std
        
        # Predict
        logits = detector(X_zs_scaled.to(device))
        probs = torch.sigmoid(logits)
        
        for p in probs:
            if p < threshold:
                # Trigger RAG
                total_cost_adaptive += (COST_ZS + COST_RAG)
                rag_triggers += 1
            else:
                # Stay Zero-Shot
                total_cost_adaptive += COST_ZS

    # Baselines
    cost_always_zs = total_samples * COST_ZS
    cost_always_rag = total_samples * COST_RAG
    
    savings = (1 - (total_cost_adaptive / cost_always_rag)) * 100
    
    print(f"--- Efficiency Report (Threshold: {threshold}) ---")
    print(f"RAG Trigger Rate: {rag_triggers/total_samples:.2%}")
    print(f"Adaptive Cost:    {total_cost_adaptive}")
    print(f"Always-RAG Cost:  {cost_always_rag}")
    print(f"Total Savings:    {savings:.2f}% vs. Always-RAG")
    
    return savings

# Run the simulation
savings_80 = simulate_system_efficiency(X_test_raw, detector, X_mean, X_std, threshold=0.8)

In [ ]:
def iterative_reflective_rag(query, k=3, max_iters=3, threshold=0.95):
    attempts = []
    
    for i in range(max_iters + 1):
        # 1. Retrieval (Offset ensures we don't see the same docs twice)
        context = None if i == 0 else retrieve_docs(query, limit=k, offset=(i-1)*k)
        
        # 2. Generation & Feature Extraction
        # is_rag flag tells the MLP to use the 'RAG-tuned' logic
        answer, raw_feats = extract_features_from_query(query, context, is_rag=(i > 0))
        
        # 3. Scaling & Confidence Prediction
        norm_feats = (raw_feats - X_mean) / X_std
        detector.eval()
        with torch.no_grad():
            conf = torch.sigmoid(detector(norm_feats.to(device))).item()
        
        attempts.append({'answer': answer, 'conf': conf, 'iter': i})
        print(f"Iter {i} | Conf: {conf:.4f} | Ans: {answer[:50]}...")

        # 4. The "Success" Break
        if conf >= threshold:
            print(f"🎯 Threshold met! Final Answer found at Iter {i}.")
            return answer, attempts
            
    # 5. Fallback: The "Most Confident" Answer
    print("Threshold never hit. Returning the most stable guess.")
    best = max(attempts, key=lambda x: x['conf'])
    return best['answer'], attempts

In [ ]:
# def final_reflective_rag_call(query, threshold=0.9):
#     answer, trajectory = iterative_reflective_rag(query, threshold=threshold)
    
#     final_conf = trajectory[-1]['conf']
    
#     if final_conf < threshold:
#         disclaimer = "\n\n(Note: I couldn't find 100% verifying evidence for this in my sources, so please use with caution.)"
#         return f"{answer}{disclaimer}"
    
#     return answer

In [ ]:
def simulate_reflective_rag_performance(X_test_raw, y_test, detector, X_mean, X_std, threshold=0.95):
    detector.eval()
    num_questions = len(X_test_raw) // 2  # 500 questions total
    
    # Categories for our Scorecard
    scorecard = {
        "Instant_Correct": 0,    # Correct at ZS, High Conf (Skipped RAG)
        "Recovered_Correct": 0,  # Wrong at ZS, High Conf at RAG (Saved by Loop)
        "Honest_Failure": 0,     # Not confident after RAG (Flagged with Disclaimer)
        "False_Confidence": 0,   # High Conf but Answer is Wrong (The remaining 4%)
        "Efficiency_Loss": 0     # Correct at ZS, but we ran RAG anyway
    }

    with torch.no_grad():
        for i in range(num_questions):
            idx_zs, idx_rag = i * 2, i * 2 + 1
            
            # 1. Scale and Predict
            feat_zs = ((X_test_raw[idx_zs] - X_mean) / X_std).to(device)
            feat_rag = ((X_test_raw[idx_rag] - X_mean) / X_std).to(device)
            
            conf_zs = torch.sigmoid(detector(feat_zs.unsqueeze(0))).item()
            conf_rag = torch.sigmoid(detector(feat_rag.unsqueeze(0))).item()
            
            correct_zs = y_test[idx_zs].item() == 1
            correct_rag = y_test[idx_rag].item() == 1

            # --- SIMULATION LOGIC ---
            
            # STEP 1: Check Zero-Shot
            if conf_zs >= threshold:
                if correct_zs:
                    scorecard["Instant_Correct"] += 1
                else:
                    scorecard["False_Confidence"] += 1
            
            # STEP 2: Trigger RAG (The "Reflective" Step)
            else:
                if conf_rag >= threshold:
                    if correct_rag:
                        scorecard["Recovered_Correct"] += 1
                    else:
                        scorecard["False_Confidence"] += 1
                else:
                    # STEP 3: Fallback (The Disclaimer Step)
                    scorecard["Honest_Failure"] += 1
                    if correct_zs and not correct_rag: 
                        # Rare case where RAG broke a correct ZS answer
                        pass 

    return scorecard

# Execute the simulation
final_results = simulate_reflective_rag_performance(X_test_raw, y_test, detector, X_mean, X_std, threshold=0.95)
final_results